# Fine-tuning do Modelo Moirai-MoE com Componentes Bayesianos para Criptomoedas

Este notebook demonstra como fazer o fine-tuning do modelo Moirai-MoE com componentes bayesianos para previsão de séries temporais de criptomoedas usando o repositório uni2ts.

## Passos deste notebook:
1. Configuração do ambiente Kaggle
2. Download e instalação do repositório uni2ts
3. Preparação dos dados de criptomoedas
4. Configuração e ajuste do modelo Moirai-MoE
5. Treinamento e monitoramento do modelo
6. Avaliação do modelo e visualização de resultados
7. Salvamento do modelo e exportação

## 1. Configuração do Ambiente Kaggle

Primeiro, vamos verificar a versão do Python e configurar o ambiente Kaggle. O Moirai-MoE é otimizado para Python 3.11, então é importante verificar a compatibilidade.

In [ ]:
import sys
print(f"Python version: {sys.version}")

# Verificar disponibilidade de GPU (recomendado para treinamento)
!nvidia-smi

### 1.1 Configuração de Diretórios

Vamos criar a estrutura de diretórios necessária para o projeto.

In [ ]:
import os

# Diretórios principais
BASE_DIR = "/kaggle/working"
REPO_DIR = os.path.join(BASE_DIR, "uni2ts")
DATA_DIR = os.path.join(BASE_DIR, "data")
OUTPUT_DIR = os.path.join(BASE_DIR, "output")
LOG_DIR = os.path.join(BASE_DIR, "logs")
PLOT_DIR = os.path.join(BASE_DIR, "plots")

# Criar diretórios
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)
os.makedirs(PLOT_DIR, exist_ok=True)

print(f"Diretórios criados:")
print(f"- Repositório: {REPO_DIR}")
print(f"- Dados: {DATA_DIR}")
print(f"- Saída: {OUTPUT_DIR}")
print(f"- Logs: {LOG_DIR}")
print(f"- Gráficos: {PLOT_DIR}")

## 2. Download e Instalação do Repositório uni2ts

Agora vamos clonar o repositório e instalar o pacote e suas dependências.

In [ ]:
# Clonar o repositório
!git clone -b moiraitst https://github.com/waldefran/uni2ts.git {REPO_DIR}

# Mudar para o diretório do repositório
%cd {REPO_DIR}

# Listar conteúdo do diretório para confirmar
!ls -la

### 2.1 Instalação das Dependências

Vamos instalar as dependências necessárias para o treinamento do modelo.

In [ ]:
# Instalar o pacote em modo de desenvolvimento e suas dependências
!pip install -e .

# Instalar dependências específicas para criptomoedas
!pip install -r requirements_crypto.txt

# Verificar instalação
!pip list | grep -E "torch|lightning|pandas|numpy|scipy|matplotlib|wandb"

### 2.2 Verificação do Ambiente

Vamos verificar se o ambiente está configurado corretamente para o treinamento do modelo bayesiano de criptomoedas.

In [ ]:
import os

# Garantir que o arquivo .env existe para evitar alertas
env_path = os.path.join(REPO_DIR, ".env")
env_content = """# Configurações de ambiente para uni2ts
PYTHONPATH=.
WANDB_API_KEY=
WANDB_PROJECT=uni2ts-crypto
WANDB_ENTITY=uni2ts
WANDB_MODE=disabled
CUDA_VISIBLE_DEVICES=0
"""

# Criar ou atualizar o arquivo .env
with open(env_path, "w") as f:
    f.write(env_content)
print(f"Arquivo .env criado/atualizado em {env_path}")

# Criar um patch temporário para o erro de importação no CLI
import sys
from types import ModuleType

# Executar o script de verificação do ambiente
!python environment_report.py

### 2.3 Verificação dos Componentes SOTA

Vamos verificar se todos os componentes necessários para o pipeline SOTA estão funcionando corretamente.

In [ ]:
# Adicionar caminho para encontrar o módulo uni2ts
import sys
import os

# Adicionar o diretório src ao sys.path
src_path = os.path.abspath('src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)

print(f"📁 Caminho src adicionado: {src_path}")

# Testar a importação dos componentes principais
import torch
import pytorch_lightning as pl
from uni2ts.model.crypto.bayesian_head import BayesianPredictionHead
from uni2ts.loss.bayesian_elbo import BayesianELBOLoss
from uni2ts.data.builder.crypto import CryptoDatasetBuilder, CryptoConfig
from uni2ts.callbacks.bayesian_uncertainty import BayesianUncertaintyMonitor
from uni2ts.callbacks.elbo_annealing import ELBOAnnealingCallback
from uni2ts.model.moirai_moe import MoiraiMoEModule  # Corrigido caminho de importação

print("Importação dos componentes SOTA realizada com sucesso!")

# Configuração e Criação do CryptoDatasetBuilder SOTA
print("🏆 CONFIGURAÇÃO SOTA - CRYPTODATASETBUILDER")
print("=" * 60)

# Verificar dados baixados no diretório correto
import glob
parquet_files = glob.glob(os.path.join(BINANCE_DATA_DIR, "**", "*.parquet"), recursive=True)
print(f"📁 Arquivos .parquet encontrados: {len(parquet_files)}")

if not parquet_files:
    raise FileNotFoundError(f"❌ Nenhum arquivo .parquet encontrado em {BINANCE_DATA_DIR}")

# Mostrar arquivos encontrados
for file in parquet_files:
    size_mb = os.path.getsize(file) / (1024*1024)
    asset_name = os.path.basename(os.path.dirname(file))
    file_name = os.path.basename(file)
    print(f"   📄 {asset_name}/{file_name} ({size_mb:.1f}MB)")

# Configuração SOTA otimizada para dados de 1 minuto
crypto_config = CryptoConfig(
    # Janelas temporais otimizadas
    context_length=1440,        # 24 horas de contexto
    prediction_length=60,       # 1 hora de predição
    
    # Features SOTA ativadas (BOOST.MD)
    unified_dataset=True,       # ✅ Dataset unificado
    anonymous_training=True,    # ✅ Treinamento anônimo
    window_normalization=True,  # ✅ Normalização por janela
    cyclical_features=True,     # ✅ Features temporais cíclicas
    
    # Parâmetros de qualidade
    min_sequence_length=1440,   # Mínimo 24h de dados
    validation_split=0.2,       # 20% validação
    dtype='float32',           # Tipo de dados otimizado
    
    # Ativos para unificação
    target_assets=assets
)

print(f"\n⚙️ Configuração SOTA:")
print(f"   Context: {crypto_config.context_length}min | Prediction: {crypto_config.prediction_length}min")
print(f"   Dataset unificado: ✅ | Anônimo: ✅ | Normalização: ✅ | Features cíclicas: ✅")
print(f"   Ativos: {len(crypto_config.target_assets)} | Dtype: {crypto_config.dtype}")

# Criar o builder
try:
    dataset_builder = CryptoDatasetBuilder(
        data_path=BINANCE_DATA_DIR,
        config=crypto_config
    )
    
    # Validar configuração (apenas se dados existem)
    data_exists = os.path.exists(BINANCE_DATA_DIR) and len(parquet_files) > 0
    if data_exists:
        is_valid = dataset_builder.validate_config(check_data_path=True)
        if not is_valid:
            raise RuntimeError("❌ Validação da configuração falhou")
        print("✅ Configuração validada com dados reais")
    else:
        print("⚠️ Validação pulada - dados não encontrados")
    
    print("✅ CryptoDatasetBuilder SOTA criado")
    
    # Construir datasets
    print("\n🏗️ Construindo datasets SOTA...")
    train_dataset, val_dataset, test_dataset = dataset_builder.build_datasets()
    
    print(f"✅ Datasets criados:")
    print(f"   🚂 Treino: {len(train_dataset)} sequências")
    print(f"   🔬 Validação: {len(val_dataset)} sequências")
    print(f"   🧪 Teste: {len(test_dataset)} sequências")
    
    # Validar formato
    sample = train_dataset[0]
    print(f"\n📊 Formato da amostra: {list(sample.keys())}")
    
    if 'target' in sample:
        print(f"   Target length: {len(sample['target'])}")
    if 'feat_dynamic_real' in sample:
        import numpy as np
        feat_array = np.array(sample['feat_dynamic_real'])
        print(f"   Features shape: {feat_array.shape}")
    
    # Informações das features SOTA
    feature_info = dataset_builder.get_feature_info()
    print(f"\n📈 Features SOTA implementadas:")
    print(f"   Numéricas: {len(feature_info['numerical_fields'])}")
    print(f"   Técnicas: {len(feature_info['technical_fields'])}")
    print(f"   Cíclicas: {len(feature_info['cyclical_fields'])}")
    print(f"   Total: {feature_info['total_features']} features")
    
    print(f"\n🎉 DATASET BUILDER SOTA PRONTO!")
    
except Exception as e:
    print(f"❌ ERRO: {e}")
    import traceback
    traceback.print_exc()
    raise

## 3. Preparação dos Dados de Criptomoedas

Agora vamos preparar os dados de criptomoedas para o treinamento do modelo. Para isso, utilizaremos o CryptoDatasetBuilder do uni2ts.

### 3.1 Download dos Dados da Binance (Estado da Arte - 1 Minuto)

Vamos baixar dados de **1 minuto (m1)** da Binance usando o `BinanceDataDownloader` otimizado. 

#### Por que dados de 1 minuto são Estado da Arte?

1. **🚀 Máxima resolução temporal**: Captura micro-movimentos e padrões intra-hora
2. **📊 Volume de dados rico**: Permite aprendizado de padrões complexos 
3. **⚡ Trading de alta frequência**: Essencial para estratégias modernas
4. **🎯 Granularidade ótima**: Balança ruído vs. informação útil
5. **🏆 Padrão da indústria**: Usado pelos melhores sistemas de trading

O script `binanceDataloader.py` foi otimizado especificamente para baixar dados de 1 minuto com máxima eficiência e estrutura de paths correta para o `CryptoDatasetBuilder`.

In [ ]:
# Download dos Dados SOTA - 1 Minuto da Binance
print("📊 DOWNLOAD DOS DADOS CRYPTO SOTA - 1 MINUTO")
print("=" * 60)

# Criar diretório para dados da Binance
BINANCE_DATA_DIR = os.path.join(DATA_DIR, "binance_data")
os.makedirs(BINANCE_DATA_DIR, exist_ok=True)

# Importar e usar a classe BinanceDataDownloader SOTA
import sys
sys.path.append(REPO_DIR)
from binanceDataloader import BinanceDataDownloader

# Lista de ativos cripto SOTA (top crypto por liquidez)
assets = ["BTCUSDT", "ETHUSDT", "BNBUSDT", "ADAUSDT"]  # Reduzido para Kaggle

print(f"⚡ Configuração SOTA:")
print(f"   Intervalo: 1 minuto (m1) - Máxima resolução temporal")
print(f"   Ativos: {assets}")
print(f"   Período: 6 meses (otimizado para Kaggle)")
print(f"   Diretório: {BINANCE_DATA_DIR}")

# Criar downloader
downloader = BinanceDataDownloader(output_dir=BINANCE_DATA_DIR)

# Download dos dados
print(f"\n🚀 Iniciando download...")
try:
    results = downloader.download_all_symbols(
        symbols=assets, 
        years_back=0.5  # 6 meses
    )
    
    # Verificar resultados
    print(f"\n📋 Resultados do download:")
    successful_downloads = 0
    
    for symbol, df in results.items():
        if df is not None and not df.empty:
            successful_downloads += 1
            
            # Verificar arquivo salvo
            expected_file = os.path.join(BINANCE_DATA_DIR, symbol, f"{symbol}_1m_0.5years.parquet")
            if os.path.exists(expected_file):
                file_size = os.path.getsize(expected_file) / (1024*1024)  # MB
                print(f"   ✅ {symbol}: {len(df):,} registros | {file_size:.1f}MB")
                
                # Validar colunas obrigatórias
                required_cols = ['open_time', 'open', 'high', 'low', 'close', 'volume']
                missing_cols = [col for col in required_cols if col not in df.columns]
                if missing_cols:
                    print(f"      ⚠️ Colunas faltando: {missing_cols}")
                else:
                    print(f"      ✅ Todas as colunas obrigatórias presentes")
            else:
                print(f"   ⚠️ {symbol}: Arquivo não encontrado em {expected_file}")
        else:
            print(f"   ❌ {symbol}: Falha no download")
    
    if successful_downloads >= 2:  # Mínimo 2 ativos para treinamento
        print(f"\n🎉 DOWNLOAD CONCLUÍDO: {successful_downloads}/{len(assets)} ativos")
        print(f"✅ Dados SOTA de 1 minuto prontos para o CryptoDatasetBuilder!")
    else:
        raise RuntimeError(f"❌ Downloads insuficientes: {successful_downloads}/{len(assets)}")

except Exception as e:
    print(f"❌ Erro no download: {e}")
    print(f"\n🔧 Tentando download simplificado...")
    
    # Fallback - criar dados de exemplo para demonstração
    import pandas as pd
    import numpy as np
    from datetime import datetime, timedelta
    
    print("📝 Criando dados de exemplo para demonstração...")
    
    # Criar dados sintéticos para cada ativo
    for asset in assets[:2]:  # Apenas 2 ativos para exemplo
        asset_dir = os.path.join(BINANCE_DATA_DIR, asset)
        os.makedirs(asset_dir, exist_ok=True)
        
        # Gerar série temporal sintética
        periods = 24 * 60 * 30  # 30 dias de dados de 1 minuto
        dates = pd.date_range(
            start=datetime.now() - timedelta(days=30),
            periods=periods,
            freq='1min'
        )
        
        # Preços sintéticos com walk aleatório
        np.random.seed(42)
        initial_price = 50000 if 'BTC' in asset else 3000
        returns = np.random.normal(0, 0.001, periods)
        prices = initial_price * np.exp(np.cumsum(returns))
        
        # Criar DataFrame sintético
        df_synthetic = pd.DataFrame({
            'open_time': dates,
            'open': prices,
            'high': prices * (1 + np.abs(np.random.normal(0, 0.005, periods))),
            'low': prices * (1 - np.abs(np.random.normal(0, 0.005, periods))),
            'close': prices,
            'volume': np.random.exponential(1000, periods),
            'quote_asset_volume': prices * np.random.exponential(1000, periods),
            'number_of_trades': np.random.poisson(100, periods),
            'taker_buy_base_asset_volume': np.random.exponential(500, periods),
            'taker_buy_quote_asset_volume': prices * np.random.exponential(500, periods)
        })
        
        # Salvar arquivo
        output_file = os.path.join(asset_dir, f"{asset}_1m_synthetic.parquet")
        df_synthetic.to_parquet(output_file, index=False)
        
        print(f"   📝 {asset}: {len(df_synthetic):,} registros sintéticos criados")
    
    print(f"⚠️ Usando dados sintéticos para demonstração")
    print(f"✅ Para dados reais, verifique conexão e execute novamente")

# Verificação final dos arquivos
import glob
final_files = glob.glob(os.path.join(BINANCE_DATA_DIR, "**", "*.parquet"), recursive=True)
print(f"\n📁 Arquivos finais encontrados: {len(final_files)}")
for file in final_files:
    size_mb = os.path.getsize(file) / (1024*1024)
    print(f"   📄 {os.path.basename(file)}: {size_mb:.1f}MB")

if len(final_files) >= 2:
    print(f"\n🎉 DADOS PRONTOS PARA O CRYPTODATASETBUILDER SOTA!")
else:
    raise RuntimeError("❌ Dados insuficientes para treinamento")

### 3.2 Verificação e Validação dos Dados Baixados

Agora vamos verificar se os dados foram baixados corretamente e analisar a estrutura dos dados de 1 minuto para garantir que estão no formato adequado para o CryptoDatasetBuilder SOTA.

### 3.2 Preparação do Dataset (Configuração SOTA)

Agora vamos preparar o dataset usando o **CryptoDatasetBuilder** com configurações de estado da arte para dados de 1 minuto:

#### Configurações SOTA Implementadas:

1. **Context Length**: 1440 minutos (24 horas) - janela temporal otimizada
2. **Prediction Length**: 60 minutos (1 hora) - horizonte de predição prático
3. **Dataset Unificado**: Todos os ativos em um dataset único para melhor generalização
4. **Treinamento Anônimo**: Remove identificadores de ativos durante o treinamento
5. **Normalização por Janela**: Foca na forma dos padrões, não na escala absoluta
6. **Features Cíclicas**: Componentes temporais (minuto, hora, dia da semana) com codificação sin/cos

Essas configurações seguem as melhores práticas para modelos transformers em séries temporais financeiras.

In [ ]:
# Importar bibliotecas necessárias
import yaml
import copy
from pathlib import Path
import os
import glob
import pandas as pd

# Carregar o arquivo de configuração
config_path = os.path.join(REPO_DIR, "configs", "crypto", "finetune_bayesian_moe.yaml")
with open(config_path, "r") as f:
    config = yaml.safe_load(f)

print("🔧 CONFIGURANDO PARÂMETROS SOTA PARA DADOS DE 1 MINUTO")
print("=" * 60)

# Ajustar configurações SOTA para dados de 1 minuto (corrigir estrutura YAML)
config["data"]["data_path"] = BINANCE_DATA_DIR
config["data"]["config"]["target_assets"] = assets

# Configurações SOTA para dados de 1 minuto (m1) - estrutura correta
config["data"]["config"]["context_length"] = 1440  # 24 horas em minutos
config["data"]["config"]["prediction_length"] = 60  # 1 hora de predição
config["data"]["config"]["unified_dataset"] = True  # Dataset unificado SOTA
config["data"]["config"]["anonymous_training"] = True  # Anonimização SOTA
config["data"]["config"]["window_normalization"] = True  # Normalização SOTA
config["data"]["config"]["cyclical_features"] = True  # Features cíclicas SOTA

# Ajustar batch size para dados de alta frequência (1 minuto)
config["train_dataloader"]["batch_size"] = 16  # Reduzido para m1 data
config["train_dataloader"]["num_workers"] = 2  # Otimizado para Kaggle
config["val_dataloader"]["batch_size"] = 16   # Consistente
config["val_dataloader"]["num_workers"] = 2   # Otimizado para Kaggle

# Ajustar parâmetros de treinamento para estado da arte
config["trainer"]["max_epochs"] = 10  # Ajustado para demonstração Kaggle
config["trainer"]["accumulate_grad_batches"] = 4  # Gradiente acumulado para eficiência
config["trainer"]["gradient_clip_val"] = 1.0  # Clipping para estabilidade

# Configurar modelo para dados de alta frequência
config["model"]["context_length"] = 1440  # Consistente com dados
config["model"]["prediction_length"] = 60  # Consistente com dados

# Ajustar loss Bayesiana para dados de 1 minuto
config["loss_func"]["kl_weight"] = 0.01  # Peso KL ajustado para m1

# Configurar callbacks para monitoramento SOTA (ajustar path do plot_dir)
for callback in config["callbacks"]:
    if callback.get("_target_") == "uni2ts.callbacks.bayesian_uncertainty.BayesianUncertaintyMonitor":
        # Adicionar plot_dir se não existir
        if "plot_dir" not in callback:
            callback["plot_dir"] = PLOT_DIR

print("✅ Configurações SOTA aplicadas:")
print(f"   📊 Context Length: {config['data']['config']['context_length']} minutos")
print(f"   🔮 Prediction Length: {config['data']['config']['prediction_length']} minutos")
print(f"   🎛️ Batch Size: {config['train_dataloader']['batch_size']}")
print(f"   🧠 Model Context: {config['model']['context_length']}")
print(f"   📈 Assets: {config['data']['config']['target_assets']}")

# Salvar configuração atualizada para referência
updated_config_path = os.path.join(OUTPUT_DIR, "finetune_bayesian_moe_sota_m1.yaml")
os.makedirs(os.path.dirname(updated_config_path), exist_ok=True)
with open(updated_config_path, "w") as f:
    yaml.dump(config, f, default_flow_style=False)

print(f"\n💾 Configuração SOTA salva em: {updated_config_path}")
print(f"\n🎯 Configuração otimizada para dados de 1 minuto - Estado da Arte!")

# Verificação Detalhada dos Dados Baixados
print("🔍 VERIFICAÇÃO DETALHADA DOS DADOS CRYPTO")
print("=" * 50)

# Verificar se BINANCE_DATA_DIR foi definido na célula anterior
if 'BINANCE_DATA_DIR' not in globals():
    # Fallback se a variável não foi definida
    BINANCE_DATA_DIR = os.path.join(DATA_DIR, "binance_data")
    print(f"⚠️ BINANCE_DATA_DIR não encontrado, usando: {BINANCE_DATA_DIR}")

# Verificar existência do diretório
if not os.path.exists(BINANCE_DATA_DIR):
    print(f"❌ Diretório não encontrado: {BINANCE_DATA_DIR}")
    print(f"🔧 Execute a célula de download dos dados primeiro!")
    raise FileNotFoundError(f"Diretório de dados não encontrado: {BINANCE_DATA_DIR}")

# Buscar todos os arquivos .parquet
parquet_files = glob.glob(os.path.join(BINANCE_DATA_DIR, "**", "*.parquet"), recursive=True)

print(f"📁 Status dos arquivos:")
print(f"   Diretório base: {BINANCE_DATA_DIR}")
print(f"   Arquivos .parquet encontrados: {len(parquet_files)}")

if len(parquet_files) == 0:
    print(f"\n❌ ERRO: Nenhum arquivo .parquet encontrado!")
    print(f"🔧 Soluções:")
    print(f"   1. Execute a célula de download dos dados")
    print(f"   2. Verifique se a conexão com a internet está funcionando")
    print(f"   3. Verifique se o diretório {BINANCE_DATA_DIR} foi criado")
    
    # Listar conteúdo do diretório para debug
    if os.path.exists(BINANCE_DATA_DIR):
        print(f"\n📋 Conteúdo do diretório {BINANCE_DATA_DIR}:")
        for item in os.listdir(BINANCE_DATA_DIR):
            item_path = os.path.join(BINANCE_DATA_DIR, item)
            if os.path.isdir(item_path):
                print(f"   📁 {item}/")
                sub_items = os.listdir(item_path)
                for sub_item in sub_items:
                    print(f"      📄 {sub_item}")
            else:
                print(f"   📄 {item}")
    
    raise FileNotFoundError("Execute o download dos dados primeiro!")

# Analisar cada arquivo encontrado
print(f"\n📊 Análise detalhada dos arquivos:")
valid_files = 0
total_records = 0

# Colunas obrigatórias para o CryptoDatasetBuilder
required_columns = ['open_time', 'open', 'high', 'low', 'close', 'volume']

for file_path in parquet_files:
    try:
        # Ler arquivo
        df = pd.read_parquet(file_path)
        
        # Extrair informações do path
        asset_name = os.path.basename(os.path.dirname(file_path))
        file_name = os.path.basename(file_path)
        file_size = os.path.getsize(file_path) / (1024*1024)  # MB
        
        print(f"\n   📄 {asset_name}/{file_name}:")
        print(f"      📊 Registros: {len(df):,}")
        print(f"      💾 Tamanho: {file_size:.1f}MB")
        print(f"      📅 Período: {df['open_time'].min()} até {df['open_time'].max()}")
        print(f"      🔢 Colunas: {len(df.columns)} ({list(df.columns)})")
        
        # Verificar colunas obrigatórias
        missing_cols = [col for col in required_columns if col not in df.columns]
        if missing_cols:
            print(f"      ⚠️ Colunas faltando: {missing_cols}")
        else:
            print(f"      ✅ Todas as colunas obrigatórias presentes")
            valid_files += 1
            total_records += len(df)
        
        # Verificar tipos de dados
        print(f"      🔧 Tipo open_time: {df['open_time'].dtype}")
        if df['open_time'].dtype != 'datetime64[ns]':
            print(f"      ⚠️ open_time não é datetime, tentando conversão...")
            
    except Exception as e:
        print(f"   ❌ Erro ao ler {file_path}: {e}")

print(f"\n📈 RESUMO DA VALIDAÇÃO:")
print(f"   Arquivos válidos: {valid_files}/{len(parquet_files)}")
print(f"   Total de registros: {total_records:,}")

if valid_files >= 2:  # Mínimo 2 ativos para treinamento
    print(f"   ✅ Dados suficientes para treinamento SOTA!")
    
    # Atualizar lista de ativos baseada nos arquivos encontrados
    assets = []
    for file_path in parquet_files:
        asset_name = os.path.basename(os.path.dirname(file_path))
        if asset_name not in assets:
            assets.append(asset_name)
    
    print(f"   🎯 Ativos disponíveis: {assets}")
    
else:
    raise RuntimeError(f"❌ Dados insuficientes: apenas {valid_files} arquivos válidos encontrados")

print(f"\n🎉 DADOS VALIDADOS E PRONTOS PARA O CRYPTODATASETBUILDER!")

In [ ]:
# Adicionar caminho para encontrar o módulo uni2ts
import sys
import os

# Adicionar o diretório src ao sys.path
src_path = os.path.abspath('src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)

print(f"📁 Caminho src adicionado: {src_path}")

# Importar e configurar o CryptoDatasetBuilder SOTA COMPLETO
from uni2ts.data.builder.crypto import CryptoDatasetBuilder, CryptoConfig

# Configuração SOTA para dados de 1 minuto
print("🏆 CONFIGURAÇÃO CRYPTO DATASET BUILDER - ESTADO DA ARTE COMPLETO")
print("=" * 65)

# Verificar se os dados foram baixados corretamente
import glob
parquet_files = glob.glob(os.path.join(BINANCE_DATA_DIR, "**", "*.parquet"), recursive=True)
print(f"📁 Arquivos .parquet encontrados: {len(parquet_files)}")

if parquet_files:
    for file in parquet_files:
        size_mb = os.path.getsize(file) / (1024*1024)
        # Extrair nome do ativo do path
        asset_name = os.path.basename(os.path.dirname(file))
        file_name = os.path.basename(file)
        print(f"   ✅ {asset_name}/{file_name} ({size_mb:.1f} MB)")
        
        # Verificar rapidamente o conteúdo do arquivo
        import pandas as pd
        try:
            df_sample = pd.read_parquet(file, nrows=5)
            print(f"      📊 Colunas: {list(df_sample.columns)}")
            print(f"      📅 Data início: {df_sample['open_time'].min()}")
            print(f"      📅 Data fim: {df_sample['open_time'].max()}")
        except Exception as e:
            print(f"      ⚠️ Erro ao ler arquivo: {e}")
else:
    raise FileNotFoundError(f"❌ ERRO: Nenhum arquivo .parquet encontrado em {BINANCE_DATA_DIR}. "
                           f"Execute a célula de download dos dados primeiro.")
    
print("\n" + "="*65)

# Configuração SOTA COMPLETA para dados de 1 minuto - TODAS AS FEATURES ATIVADAS
print("🎯 CONFIGURAÇÃO SOTA COMPLETA PARA DADOS DE 1 MINUTO:")

crypto_config = CryptoConfig(
    # Parâmetros temporais otimizados para dados de 1 minuto
    context_length=1440,        # 24 horas (1440 min) - janela temporal SOTA
    prediction_length=60,       # 1 hora (60 min) - horizonte prático para trading
    
    # TODAS AS CONFIGURAÇÕES SOTA ATIVADAS (BOOST.MD)
    unified_dataset=True,       # Dataset unificado - melhora generalização ✅
    anonymous_training=True,    # Anonimização - força padrões universais ✅
    window_normalization=True,  # Normalização por janela - foca na forma ✅
    cyclical_features=True,     # Features temporais (min, hora, dia semana) ✅
    
    # Filtros de qualidade SOTA
    min_sequence_length=1440,   # Mínimo 24h de dados contínuos
    
    # Divisão dos dados otimizada
    validation_split=0.2,       # 20% para validação
    
    # Ativos definidos no download
    target_assets=assets
)

print(f"   ⏱️  Context Length: {crypto_config.context_length} minutos (24 horas)")
print(f"   🔮 Prediction Length: {crypto_config.prediction_length} minutos (1 hora)")
print(f"   🎭 Dataset Unificado: {crypto_config.unified_dataset} ✅ SOTA")
print(f"   👤 Treinamento Anônimo: {crypto_config.anonymous_training} ✅ SOTA")
print(f"   📊 Normalização por Janela: {crypto_config.window_normalization} ✅ SOTA")
print(f"   🔄 Features Cíclicas: {crypto_config.cyclical_features} ✅ SOTA")
print(f"   📏 Seq. Mínima: {crypto_config.min_sequence_length} minutos")
print(f"   📈 Ativos: {len(crypto_config.target_assets)} símbolos")
print(f"   📁 Data Dir: {BINANCE_DATA_DIR}")

# Criar o CryptoDatasetBuilder SOTA COMPLETO com configuração avançada
print(f"\n🔧 Criando CryptoDatasetBuilder SOTA COMPLETO...")

# CRUCIAL: Inicializar as variáveis que serão utilizadas nas próximas células
train_dataset = None
val_dataset = None
test_dataset = None

try:
    dataset_builder = CryptoDatasetBuilder(
        data_path=BINANCE_DATA_DIR,
        config=crypto_config
    )
    
    print("✅ CryptoDatasetBuilder SOTA criado com sucesso!")
    print("🏆 TODAS as features SOTA ativadas:")
    print("   ✅ Dataset unificado (BOOST.MD)")
    print("   ✅ Anonimização (BOOST.MD)")
    print("   ✅ Normalização por janela (BOOST.MD)")
    print("   ✅ Features cíclicas sin/cos (BOOST.MD)")
    print("   ✅ Features técnicas avançadas")
    print("   ✅ Engenharia de features completa")
    print("   ✅ Split temporal inteligente")
    print("   ✅ Compatibilidade Moirai-MoE")
    
    # Validar estrutura dos dados
    print("\n🔍 Validando estrutura dos dados SOTA...")
    
    # Tentar criar datasets SOTA - CRUCIAL: Atribuir às variáveis globais
    print("📊 Criando datasets SOTA de treino, validação e teste...")
    train_dataset, val_dataset, test_dataset = dataset_builder.build_datasets()
    
    # Verificar se os datasets foram criados corretamente
    if train_dataset is None or val_dataset is None or test_dataset is None:
        raise RuntimeError("❌ ERRO: Um ou mais datasets retornaram None")
    
    print(f"✅ DATASETS SOTA CRIADOS COM SUCESSO:")
    print(f"   🚂 Treino: {len(train_dataset)} amostras")
    print(f"   🔬 Validação: {len(val_dataset)} amostras") 
    print(f"   🧪 Teste: {len(test_dataset)} amostras")
    print(f"   📊 Total: {len(train_dataset) + len(val_dataset) + len(test_dataset)} amostras")
    
    # Testar uma amostra para validar formato SOTA
    print(f"\n🔬 Testando formato SOTA de uma amostra...")
    sample = train_dataset[0]
    print(f"   📋 Chaves HuggingFace: {list(sample.keys())}")
    
    if 'target' in sample:
        target_shape = sample['target'].shape if hasattr(sample['target'], 'shape') else len(sample['target'])
        print(f"   📊 Target shape: {target_shape}")
    if 'future_target' in sample:
        future_shape = sample['future_target'].shape if hasattr(sample['future_target'], 'shape') else len(sample['future_target'])
        print(f"   🔮 Future target shape: {future_shape}")
    if 'start' in sample:
        print(f"   📅 Start timestamp: {sample['start']}")
    if 'freq' in sample:
        print(f"   🕐 Frequency: {sample['freq']}")
    
    # Mostrar informações das features SOTA
    feature_info = dataset_builder.get_feature_info()
    print(f"\n📊 INFORMAÇÕES DAS FEATURES SOTA:")
    print(f"   💰 Price/Volume fields: {len(feature_info['price_volume_fields'])} campos")
    print(f"   🔄 Cyclical features: {len(feature_info['cyclical_features'])} campos")
    print(f"   📈 Technical features: {len(feature_info['technical_features'])} campos")
    print(f"   🎯 Normalização: {feature_info['normalization']}")
    print(f"   👤 Treinamento anônimo: {feature_info['anonymous_training']}")
    
    print(f"\n🎉 DATASET BUILDER SOTA COMPLETO CONFIGURADO E VALIDADO!")
    print(f"🚀 Pronto para treinamento com TODAS as features SOTA ativadas!")
    print(f"🏆 Estado da Arte em processamento de dados crypto para ML!")
    
    # Verificação final crucial para próximas células
    print(f"\n🔍 VERIFICAÇÃO FINAL DOS DATASETS SOTA:")
    print(f"   train_dataset type: {type(train_dataset)}")
    print(f"   val_dataset type: {type(val_dataset)}")
    print(f"   test_dataset type: {type(test_dataset)}")
    print(f"   ✅ Todos os datasets SOTA estão disponíveis para as próximas células")
    
except Exception as e:
    print(f"❌ ERRO ao criar CryptoDatasetBuilder: {e}")
    print(f"💡 Diagnosticando problema...")
    
    # Debug detalhado para diagnóstico
    print(f"🔍 Debug - Verificações SOTA:")
    print(f"   📁 BINANCE_DATA_DIR existe: {os.path.exists(BINANCE_DATA_DIR)}")
    print(f"   📊 Arquivos .parquet: {len(glob.glob(os.path.join(BINANCE_DATA_DIR, '**', '*.parquet'), recursive=True))}")
    print(f"   🎯 Assets definidos: {assets}")
    print(f"   🔧 Config: {crypto_config}")
    
    # Verificar se é problema de importação
    import traceback
    print(f"\n📋 TRACEBACK COMPLETO:")
    traceback.print_exc()
    
    # Re-raise o erro para não mascarar o problema
    raise

### 3.3 Criação dos DataLoaders

Agora vamos criar os DataLoaders para o treinamento do modelo.

In [ ]:
# Importar bibliotecas necessárias
from torch.utils.data import DataLoader

# Validar que a configuração foi carregada corretamente
print("🔧 VALIDANDO CONFIGURAÇÃO DOS DATALOADERS")
print("=" * 50)

# Verificar se as chaves necessárias existem na configuração
required_keys = ["train_dataloader", "val_dataloader"]
missing_keys = [key for key in required_keys if key not in config]

if missing_keys:
    raise KeyError(f"❌ ERRO: Chaves faltando na configuração: {missing_keys}. "
                   f"Verifique se o arquivo YAML foi carregado corretamente.")

# Verificar se as subchaves necessárias existem
train_dl_config = config["train_dataloader"]
val_dl_config = config["val_dataloader"]

if "batch_size" not in train_dl_config:
    raise KeyError("❌ ERRO: 'batch_size' não encontrado em train_dataloader")
if "batch_size" not in val_dl_config:
    raise KeyError("❌ ERRO: 'batch_size' não encontrado em val_dataloader")

print("✅ Configurações de DataLoader validadas:")
print(f"   🚂 Train batch_size: {train_dl_config['batch_size']}")
print(f"   🔬 Val batch_size: {val_dl_config['batch_size']}")

# Parâmetros para os DataLoaders (estrutura validada do YAML)
batch_size = train_dl_config["batch_size"]
num_workers = train_dl_config.get("num_workers", 4)

# Otimizações de performance SOTA
prefetch_factor = 2 if num_workers > 0 else None
persistent_workers = num_workers > 0

print(f"   ⚙️ Num workers: {num_workers}")
print(f"   🚀 Prefetch factor: {prefetch_factor}")
print(f"   🔄 Persistent workers: {persistent_workers}")

# Verificar se os datasets existem (corrigir para usar globals())
datasets_missing = []
if 'train_dataset' not in globals() or train_dataset is None:
    datasets_missing.append('train_dataset')
if 'val_dataset' not in globals() or val_dataset is None:
    datasets_missing.append('val_dataset')
if 'test_dataset' not in globals() or test_dataset is None:
    datasets_missing.append('test_dataset')

if datasets_missing:
    raise RuntimeError(f"❌ ERRO: Os seguintes datasets não foram criados: {datasets_missing}. "
                      f"Execute a célula '3.2 Preparação do Dataset' primeiro e verifique se não há erros.")

print(f"✅ Datasets validados:")
print(f"   🚂 Train: {len(train_dataset)} amostras")
print(f"   🔬 Val: {len(val_dataset)} amostras")
print(f"   🧪 Test: {len(test_dataset)} amostras")

# Criação dos DataLoaders SOTA
print("🔧 CRIANDO DATALOADERS SOTA")
print("=" * 40)

# Validar que os datasets foram criados
if 'train_dataset' not in globals() or train_dataset is None:
    raise RuntimeError("❌ train_dataset não foi criado. Execute a célula de preparação dos datasets primeiro.")
if 'val_dataset' not in globals() or val_dataset is None:
    raise RuntimeError("❌ val_dataset não foi criado. Execute a célula de preparação dos datasets primeiro.")

print(f"✅ Datasets validados:")
print(f"   🚂 Train: {len(train_dataset)} sequências")
print(f"   🔬 Val: {len(val_dataset)} sequências")

# Configurações SOTA para DataLoaders (otimizadas para crypto)
train_config = {
    "batch_size": 16,           # Batch pequeno para estabilidade Bayesiana
    "shuffle": True,            # Embaralhar para treinamento
    "num_workers": 4,           # Workers para paralelização
    "pin_memory": True,         # Memory pinning para GPU
    "drop_last": True,          # Drop último batch incompleto
    "persistent_workers": True   # Manter workers vivos
}

val_config = {
    "batch_size": 32,           # Batch maior para validação (mais eficiente)
    "shuffle": False,           # Não embaralhar validação
    "num_workers": 4,
    "pin_memory": True,
    "persistent_workers": True
}

print(f"⚙️ Configurações SOTA:")
print(f"   Train batch: {train_config['batch_size']} | Val batch: {val_config['batch_size']}")
print(f"   Workers: {train_config['num_workers']} | Pin memory: {train_config['pin_memory']}")

# Criar DataLoaders
try:
    train_loader = DataLoader(train_dataset, **train_config)
    val_loader = DataLoader(val_dataset, **val_config)
    
    print("✅ DataLoaders criados com sucesso")
    
    # Testar um batch para validação
    print("\n🔬 Testando batch de treinamento...")
    sample_batch = next(iter(train_loader))
    
    print(f"   Batch keys: {list(sample_batch.keys())}")
    for key, value in sample_batch.items():
        if hasattr(value, 'shape'):
            print(f"   {key} shape: {value.shape}")
        elif isinstance(value, list):
            print(f"   {key} length: {len(value)}")
    
    print("\n🎉 DATALOADERS SOTA PRONTOS PARA TREINAMENTO!")
    
except Exception as e:
    print(f"❌ Erro ao criar DataLoaders: {e}")
    import traceback
    traceback.print_exc()
    raise

### 3.4 Análise Exploratória dos Dados

Vamos analisar brevemente os dados para entender melhor a estrutura e características dos nossos dados de treinamento.

In [ ]:
# Importar bibliotecas de visualização
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

print("🔍 ANÁLISE EXPLORATÓRIA - DADOS SOTA")
print("=" * 50)

# Configurar matplotlib para ambiente
import matplotlib
matplotlib.use('Agg')  # Backend não-interativo para Kaggle

# Obter uma amostra do dataset
sample_batch = next(iter(train_loader))

# Analisar estrutura do batch
print(f"📊 Estrutura do batch:")
print(f"   🔑 Keys: {list(sample_batch.keys())}")

for key, value in sample_batch.items():
    if hasattr(value, 'shape'):
        print(f"   📏 {key} shape: {value.shape}")
    elif isinstance(value, list):
        print(f"   📋 {key} length: {len(value[0]) if value else 0}")

# Analisar dados temporais
if 'target' in sample_batch:
    targets = sample_batch['target']
    print(f"\n📈 Análise temporal:")
    print(f"   📊 Target shape: {targets.shape}")
    print(f"   ⏱️ Janela temporal: {targets.shape[1]} minutos")
    print(f"   🎯 Features: {targets.shape[2] if len(targets.shape) > 2 else 1}")

# Features dinâmicas
if 'feat_dynamic_real' in sample_batch:
    features = sample_batch['feat_dynamic_real']
    print(f"   🔧 Features dinâmicas shape: {features.shape}")
    print(f"   📊 Total features: {features.shape[2] if len(features.shape) > 2 else features.shape[1]}")

# Visualização básica
print(f"\n📈 Visualização da primeira amostra:")

try:
    # Extrair primeira amostra
    if 'target' in sample_batch:
        sample_target = sample_batch['target'][0].detach().numpy()
        
        # Plotar apenas o preço principal (primeira feature)
        plt.figure(figsize=(12, 6))
        
        if len(sample_target.shape) > 1:
            # Múltiplas features - plotar a primeira (geralmente preço de fechamento)
            plt.plot(sample_target[:, 0], label='Preço Principal (norm.)', linewidth=1.5)
            plt.title('Série Temporal - Preço Normalizado (1 minuto)')
            plt.xlabel('Tempo (minutos)')
            plt.ylabel('Valor Normalizado')
            plt.grid(True, alpha=0.3)
            plt.legend()
            
            # Estatísticas básicas
            print(f"   📊 Estatísticas do preço:")
            print(f"      Min: {sample_target[:, 0].min():.4f}")
            print(f"      Max: {sample_target[:, 0].max():.4f}")
            print(f"      Média: {sample_target[:, 0].mean():.4f}")
            print(f"      Std: {sample_target[:, 0].std():.4f}")
        else:
            # Uma única feature
            plt.plot(sample_target, label='Target', linewidth=1.5)
            plt.title('Série Temporal - Target')
        
        # Salvar plot
        plt.tight_layout()
        plt.savefig(f'{PLOT_DIR}/exploratory_timeseries.png', dpi=150, bbox_inches='tight')
        plt.close()
        
        print(f"   ✅ Gráfico salvo em: {PLOT_DIR}/exploratory_timeseries.png")
    
    # Análise de distribuição
    if 'feat_dynamic_real' in sample_batch:
        features_data = sample_batch['feat_dynamic_real'][0].detach().numpy()
        
        plt.figure(figsize=(10, 6))
        
        if len(features_data.shape) > 1:
            # Plot histogram das primeiras features
            n_features = min(features_data.shape[1], 3)
            for i in range(n_features):
                plt.subplot(1, n_features, i+1)
                plt.hist(features_data[:, i], bins=50, alpha=0.7, density=True)
                plt.title(f'Feature {i+1}')
                plt.xlabel('Valor')
                plt.ylabel('Densidade')
                plt.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.savefig(f'{PLOT_DIR}/features_distribution.png', dpi=150, bbox_inches='tight')
        plt.close()
        
        print(f"   ✅ Distribuição salva em: {PLOT_DIR}/features_distribution.png")

    print(f"\n🎉 Análise exploratória concluída!")
    print(f"📁 Visualizações salvas em: {PLOT_DIR}/")

except Exception as e:
    print(f"⚠️ Erro na visualização: {e}")
    print("📊 Continuando sem gráficos...")

# Info do dataset builder
if 'dataset_builder' in globals():
    feature_info = dataset_builder.get_feature_info()
    print(f"\n📊 Informações das features SOTA:")
    print(f"   🔢 Numéricas: {len(feature_info['numerical_fields'])}")
    print(f"   📈 Técnicas: {len(feature_info['technical_fields'])}")
    print(f"   🔄 Cíclicas: {len(feature_info['cyclical_fields'])}")
    print(f"   🎯 Total: {feature_info['total_features']} features")
    print(f"   📋 Normalização: {feature_info['normalization']}")

print(f"\n✅ DADOS PRONTOS PARA TREINAMENTO SOTA!")

## 4. Configuração e Ajuste do Modelo Moirai-MoE

Agora vamos configurar o modelo Moirai-MoE com componentes bayesianos para o fine-tuning.

### 4.1 Configuração do Modelo

Vamos criar o modelo Moirai-MoE com a configuração do arquivo YAML.

In [ ]:
# Configuração do Modelo Moirai-MoE SOTA
print("🤖 CONFIGURAÇÃO MODELO MOIRAI-MOE SOTA")
print("=" * 50)

import torch
import lightning as L

# Verificar dispositivo
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"📱 Dispositivo: {device}")

# Importar módulo Lightning SOTA do projeto
try:
    # Import do script de fine-tuning SOTA
    import sys
    import os
    
    # Adicionar caminho dos scripts
    scripts_path = os.path.join(REPO_DIR, 'scripts', 'crypto')
    sys.path.insert(0, scripts_path)
    
    from finetune_model import MoiraiBayesianLightningModule
    print("✅ MoiraiBayesianLightningModule importado com sucesso")
    
except ImportError as e:
    print(f"❌ Erro na importação: {e}")
    print("🔧 Tentando importação alternativa...")
    
    # Fallback - criar módulo básico
    class MoiraiBayesianLightningModule(L.LightningModule):
        def __init__(self, config):
            super().__init__()
            self.config = config
            print("⚠️ Usando implementação fallback do módulo Lightning")
        
        def forward(self, x):
            return x
        
        def training_step(self, batch, batch_idx):
            return torch.tensor(0.0, requires_grad=True)
        
        def validation_step(self, batch, batch_idx):
            return torch.tensor(0.0)
        
        def configure_optimizers(self):
            return torch.optim.Adam(self.parameters(), lr=1e-4)

# Extrair configurações do YAML carregado
model_config = {
    "context_length": 1440,        # 24 horas
    "prediction_length": 60,       # 1 hora
    "d_model": 512,               # Dimensão do modelo
    "num_heads": 8,               # Cabeças de atenção
    "num_layers": 6,              # Camadas do transformer
    "patch_size": 8,              # Tamanho do patch
    "num_samples": 100,           # Amostras para inferência
}

# Configurações Bayesianas SOTA
bayesian_config = {
    "mc_dropout_rate": 0.15,
    "num_mc_samples": 100,
    "use_variational_weights": True,
    "use_temporal_attention": True,
    "student_t_df": 4.0,
    "confidence_levels": [0.8, 0.9, 0.95, 0.99]
}

# Configurações de loss Bayesiana
loss_config = {
    "likelihood_weight": 1.0,
    "kl_weight": 1e-4,
    "kl_annealing": True,
    "kl_annealing_epochs": 50,
    "use_student_t": True,
    "student_t_df_min": 2.1
}

print(f"⚙️ Configurações do modelo:")
print(f"   📏 Context: {model_config['context_length']} | Prediction: {model_config['prediction_length']}")
print(f"   🧠 D-model: {model_config['d_model']} | Heads: {model_config['num_heads']}")
print(f"   🔬 MC Dropout: {bayesian_config['mc_dropout_rate']} | MC Samples: {bayesian_config['num_mc_samples']}")

# Obter dimensões do dataset
sample_batch = next(iter(train_loader))
print(f"\n📊 Dimensões do batch:")
for key, value in sample_batch.items():
    if hasattr(value, 'shape'):
        print(f"   {key}: {value.shape}")

# Determinar número de features
if 'feat_dynamic_real' in sample_batch:
    features_tensor = sample_batch['feat_dynamic_real'][0]
    if len(features_tensor.shape) > 1:
        num_features = features_tensor.shape[1]
    else:
        num_features = 1
    print(f"   🎯 Features detectadas: {num_features}")
else:
    num_features = len(dataset_builder.get_feature_info()['numerical_fields'])
    print(f"   🎯 Features do builder: {num_features}")

# Criar configuração completa
complete_config = {
    **model_config,
    **bayesian_config,
    **loss_config,
    "num_features": num_features,
    "batch_size": 16,
    "learning_rate": 1e-4,
    "weight_decay": 1e-5
}

# Criar modelo Lightning SOTA
try:
    model = MoiraiBayesianLightningModule(complete_config)
    print("✅ Modelo Moirai-MoE Bayesiano SOTA criado")
    
    # Contar parâmetros
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    
    print(f"📊 Parâmetros do modelo:")
    print(f"   📈 Total: {total_params:,}")
    print(f"   🎯 Treináveis: {trainable_params:,}")
    
    print(f"\n🎉 MODELO SOTA CONFIGURADO E PRONTO!")
    
except Exception as e:
    print(f"❌ Erro ao criar modelo: {e}")
    import traceback
    traceback.print_exc()
    raise

### 4.2 Configuração de Callbacks

Vamos configurar os callbacks para monitorar o treinamento do modelo.

In [ ]:
# Importar bibliotecas necessárias
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping, LearningRateMonitor
from uni2ts.callbacks.bayesian_uncertainty import BayesianUncertaintyMonitor
from uni2ts.callbacks.elbo_annealing import ELBOAnnealingCallback

# Lista para armazenar os callbacks
callbacks = []

# Adicionar callbacks a partir da configuração
for callback_config in config["callbacks"]:
    callback_class_name = callback_config["class"]
    callback_args = callback_config.get("args", {})
    
    if callback_class_name == "ModelCheckpoint":
        callback = ModelCheckpoint(
            dirpath=os.path.join(OUTPUT_DIR, "checkpoints"),
            **callback_args
        )
    elif callback_class_name == "EarlyStopping":
        callback = EarlyStopping(**callback_args)
    elif callback_class_name == "LearningRateMonitor":
        callback = LearningRateMonitor(**callback_args)
    elif callback_class_name == "BayesianUncertaintyMonitor":
        callback = BayesianUncertaintyMonitor(**callback_args)
    elif callback_class_name == "ELBOAnnealingCallback":
        callback = ELBOAnnealingCallback(**callback_args)
    else:
        print(f"Callback não reconhecido: {callback_class_name}")
        continue
    
    callbacks.append(callback)
    print(f"Callback adicionado: {callback_class_name}")

print(f"\nTotal de callbacks configurados: {len(callbacks)}")

### 4.3 Configuração do Logger

Vamos configurar o logger para monitorar métricas durante o treinamento.

In [ ]:
# Importar bibliotecas necessárias
from pytorch_lightning.loggers import CSVLogger, TensorBoardLogger

# Configurar CSVLogger
csv_logger = CSVLogger(
    save_dir=LOG_DIR,
    name="crypto_finetune",
    version=None
)

# Configurar TensorBoardLogger
tb_logger = TensorBoardLogger(
    save_dir=LOG_DIR,
    name="crypto_finetune_tb",
    version=None
)

# Lista de loggers
loggers = [csv_logger, tb_logger]

# Adicionar WandbLogger se desejado (opcional)
use_wandb = False
if use_wandb:
    try:
        from pytorch_lightning.loggers import WandbLogger
        import wandb
        
        # Inicializar WandbLogger
        wandb_logger = WandbLogger(
            project="uni2ts-crypto",
            name="finetune-bayesian-moe",
            save_dir=LOG_DIR
        )
        loggers.append(wandb_logger)
        print("WandbLogger configurado com sucesso!")
    except ImportError:
        print("WandbLogger não pôde ser configurado. Pacote wandb não encontrado.")

print(f"\nTotal de loggers configurados: {len(loggers)}")

## 5. Treinamento e Monitoramento do Modelo

Agora vamos treinar o modelo usando o PyTorch Lightning.

### 5.1 Configuração do Trainer

Vamos configurar o Trainer do PyTorch Lightning.

In [ ]:
# Importar bibliotecas necessárias
import pytorch_lightning as pl
import random
import numpy as np

# Seed e determinismo SOTA
pl.seed_everything(42, workers=True)
torch.backends.cudnn.benchmark = True

# Configuração do Trainer
trainer_config = config["trainer"]

# Adicionar configurações específicas do Kaggle
trainer_config["accelerator"] = "gpu" if torch.cuda.is_available() else "cpu"
trainer_config["devices"] = 1
trainer_config["default_root_dir"] = OUTPUT_DIR

# Precision mista para otimização SOTA
trainer_config["precision"] = "16-mixed" if torch.cuda.is_available() else 32

# Criar o Trainer
trainer = pl.Trainer(
    logger=loggers,
    callbacks=callbacks,
    **trainer_config
)

print(f"Trainer configurado com acelerador: {trainer_config['accelerator']}")
print(f"Precision: {trainer_config['precision']}")
print(f"Número máximo de épocas: {trainer_config['max_epochs']}")
print(f"Seed configurado para reprodutibilidade: 42")

### 5.2 Criação do Lightning Module

Vamos criar o Lightning Module para treinamento do modelo.

In [ ]:
# Importar bibliotecas necessárias
import pytorch_lightning as pl
import sys
from pathlib import Path

# Garantir que scripts/ esteja no path para importação
scripts_path = os.path.join(REPO_DIR, "scripts")
if scripts_path not in sys.path:
    sys.path.append(scripts_path)

# Importar o módulo Lightning já existente no projeto em vez de redefini-lo
try:
    # Tentar importação padrão
    from crypto.finetune_model import MoiraiBayesianLightningModule
except ImportError:
    # Importação alternativa com path completo
    sys.path.append(os.path.join(REPO_DIR))
    from scripts.crypto.finetune_model import MoiraiBayesianLightningModule

# Criar configuração para o módulo Lightning
lightning_config = {
    "model": {
        "pretrained_model_name_or_path": model_config["args"].get("pretrained_model_name_or_path", ""),
        "prediction_length": model_args.get("prediction_length", 24),
        "context_length": context_length,
        "patch_size": model_args.get("patch_size", 1),
        "num_samples": model_args.get("num_samples", 100),
        "prediction_head": {
            "_target_": "uni2ts.model.crypto.bayesian_head.BayesianPredictionHead",
            "input_size": model_args.get("d_model", 512),
            "hidden_size": model_args.get("d_model", 512),
            "output_size": target_dim,
            "dropout": model_args.get("dropout", 0.1),
            "prior_scale": loss_args.get("prior_scale", 1.0),
        },
        "freeze_backbone": model_args.get("freeze_backbone", False)
    },
    "loss_func": {
        "_target_": "uni2ts.loss.bayesian_elbo.BayesianELBOLoss",
        "kl_weight": loss_args.get("kl_weight", 1.0),
        "reduction": loss_args.get("reduction", "mean")
    },
    "optimizer": {
        "lr": optim_config["args"].get("lr", 1e-4),
        "weight_decay": optim_config["args"].get("weight_decay", 1e-5)
    }
}

# Criar o Lightning Module usando a classe existente no projeto
pl_module = MoiraiBayesianLightningModule(lightning_config)

print("Lightning Module configurado com sucesso usando a classe existente MoiraiBayesianLightningModule!")
print(f"Isso garante alinhamento com o pipeline SOTA do projeto uni2ts.")

### 5.3 Treinamento do Modelo

Agora vamos treinar o modelo com os dados de criptomoedas.

In [ ]:
# Preparar dados no formato esperado pelo módulo Lightning
# Converter os PyTorch DataLoaders para o formato esperado pelo MoiraiBayesianLightningModule
def convert_batch_format(batch):
    """Converter formato de batch se necessário"""
    if 'x' in batch and 'y' in batch:
        # Converter do formato do notebook para o formato esperado pelo modelo
        return {
            'past_target': batch['x'],
            'past_observed_target': torch.ones_like(batch['x']),
            'future_target': batch['y']
        }
    return batch

class BatchConverterDataLoader:
    def __init__(self, dataloader):
        self.dataloader = dataloader
        
    def __iter__(self):
        for batch in self.dataloader:
            yield convert_batch_format(batch)
            
    def __len__(self):
        return len(self.dataloader)

# Envolver os DataLoaders para garantir compatibilidade
train_loader_wrapped = BatchConverterDataLoader(train_loader)
val_loader_wrapped = BatchConverterDataLoader(val_loader)

# Treinar o modelo
print("Iniciando treinamento...")
trainer.fit(pl_module, train_loader_wrapped, val_loader_wrapped)
print("Treinamento concluído!")

### 5.4 Avaliação do Modelo no Conjunto de Teste

In [ ]:
# Envolver o dataloader de teste com o conversor
test_loader_wrapped = BatchConverterDataLoader(test_loader)

# Avaliar o modelo no conjunto de teste
print("Avaliando modelo no conjunto de teste...")
test_results = trainer.test(pl_module, test_loader_wrapped)
print(f"Resultados do teste: {test_results}")

## 6. Avaliação do Modelo e Visualização de Resultados

Vamos visualizar as métricas e resultados do treinamento.

### 6.1 Visualização de Métricas de Treinamento

In [ ]:
# Importar bibliotecas necessárias
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import glob

# Carregar as métricas do treinamento com checagem dinâmica
metrics_base = os.path.join(LOG_DIR, "crypto_finetune")
versions = sorted(glob.glob(os.path.join(metrics_base, "version_*")), key=os.path.getmtime)
metrics_path = os.path.join(versions[-1], "metrics.csv") if versions else None

if metrics_path and os.path.exists(metrics_path):
    metrics_df = pd.read_csv(metrics_path)
    
    # Filtrar métricas relevantes
    train_loss = metrics_df[metrics_df['train/loss_epoch'].notna()][['epoch', 'train/loss_epoch']]
    val_loss = metrics_df[metrics_df['val/loss'].notna()][['epoch', 'val/loss']]
    
    # Configurar o plot
    plt.figure(figsize=(12, 6))
    sns.lineplot(data=train_loss, x='epoch', y='train/loss_epoch', label='Train Loss')
    sns.lineplot(data=val_loss, x='epoch', y='val/loss', label='Validation Loss')
    plt.title('Training and Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss (ELBO)')
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.show()
    
    print(f"Métricas carregadas de: {metrics_path}")
else:
    print(f"Nenhum metrics.csv encontrado em {metrics_base}")
    if versions:
        print(f"Versões disponíveis: {[os.path.basename(v) for v in versions]}")
    else:
        print("Nenhuma versão de log encontrada")

### 6.2 Visualização de Predições com Incerteza

Vamos visualizar algumas predições do modelo com intervalos de confiança.

In [ ]:
# Importar bibliotecas necessárias
import torch
from uni2ts.distribution.student_t import StudentT
import numpy as np
import matplotlib.pyplot as plt

# Função para visualizar predições com intervalos de confiança
def visualize_predictions_with_uncertainty(model, dataloader, num_samples=5):
    model.eval()
    samples = []
    with torch.no_grad():
        for batch in dataloader:
            if len(samples) >= num_samples:
                break
                
            # Converter formato do batch se necessário
            if not isinstance(batch, dict) or ('x' in batch and 'y' in batch):
                x = batch['x']
                y = batch['y']
                # Converter para formato esperado pelo modelo
                model_batch = {
                    'past_target': x,
                    'past_observed_target': torch.ones_like(x),
                    'future_target': y
                }
            else:
                model_batch = batch
                x = batch.get('past_target', batch.get('target'))
                y = batch.get('future_target')
            
            # Obter predições
            prediction_output = model(model_batch)
            
            # Para cada amostra no batch
            for i in range(min(len(x), num_samples - len(samples))):
                # Extrair parâmetros da distribuição Student-T
                loc = prediction_output.loc[i].cpu().numpy()
                scale = prediction_output.scale[i].cpu().numpy()
                df = prediction_output.df[i].cpu().numpy()
                
                # Calcular intervalos de confiança
                # Criar distribuição Student-T
                dist = StudentT(loc=torch.tensor(loc), scale=torch.tensor(scale), df=torch.tensor(df))
                
                # Calcular quantis para intervalos de confiança
                lower_95 = dist.icdf(torch.tensor(0.025)).cpu().numpy()
                upper_95 = dist.icdf(torch.tensor(0.975)).cpu().numpy()
                
                # Guardar os dados para visualização
                samples.append({
                    'x': x[i].cpu().numpy(),
                    'y': y[i].cpu().numpy(),
                    'loc': loc,
                    'lower_95': lower_95,
                    'upper_95': upper_95
                })
    
    # Visualizar as predições
    fig, axs = plt.subplots(len(samples), 1, figsize=(12, 5*len(samples)))
    if len(samples) == 1:
        axs = [axs]
    
    for i, sample in enumerate(samples):
        # Obter dados
        x_data = sample['x'][:, 0]  # Assumindo que a primeira feature é o target
        y_data = sample['y']
        loc = sample['loc']
        lower_95 = sample['lower_95']
        upper_95 = sample['upper_95']
        
        # Plotar série histórica
        axs[i].plot(range(len(x_data)), x_data, 'b-', label='Histórico')
        
        # Plotar valores reais
        forecast_start = len(x_data)
        axs[i].plot(range(forecast_start, forecast_start + len(y_data)), y_data, 'g-', label='Real')
        
        # Plotar previsão e intervalo de confiança
        axs[i].plot(range(forecast_start, forecast_start + len(loc)), loc, 'r-', label='Previsão')
        axs[i].fill_between(
            range(forecast_start, forecast_start + len(loc)),
            lower_95, upper_95,
            color='r', alpha=0.2, label='IC 95%'
        )
        
        # Configurar gráfico
        axs[i].set_title(f'Amostra {i+1}: Previsão com Intervalo de Confiança')
        axs[i].set_xlabel('Tempo')
        axs[i].set_ylabel('Valor')
        axs[i].legend()
        axs[i].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    return samples

# Visualizar predições no conjunto de teste
samples = visualize_predictions_with_uncertainty(pl_module, test_loader, num_samples=3)

### 6.3 Análise de Métricas Bayesianas

In [ ]:
# Importar bibliotecas necessárias
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Função para calcular métricas de incerteza
def calculate_uncertainty_metrics(samples):
    results = []
    
    for i, sample in enumerate(samples):
        # Obter dados
        y_true = sample['y']
        y_pred = sample['loc']
        lower_95 = sample['lower_95']
        upper_95 = sample['upper_95']
        
        # Calcular métricas básicas
        mae = mean_absolute_error(y_true, y_pred)
        rmse = np.sqrt(mean_squared_error(y_true, y_pred))
        
        # Calcular largura média do intervalo de confiança
        ci_width = np.mean(upper_95 - lower_95)
        
        # Calcular cobertura do intervalo de confiança (% de pontos dentro do IC)
        in_interval = np.logical_and(y_true >= lower_95, y_true <= upper_95)
        coverage = np.mean(in_interval) * 100
        
        # Calcular CRPS (Continuous Ranked Probability Score) aproximado
        # Simplificação: usamos apenas média e desvio padrão para aproximar
        scale = (upper_95 - lower_95) / (2 * 1.96)  # Aproximação do desvio padrão
        crps_approx = np.mean(scale * (np.sqrt(2/np.pi) - 2 * norm.pdf((y_true - y_pred) / scale) - 
                                      (y_true - y_pred) / scale * (2 * norm.cdf((y_true - y_pred) / scale) - 1)))
        
        results.append({
            'Sample': i+1,
            'MAE': mae,
            'RMSE': rmse,
            'CI Width': ci_width,
            'Coverage (%)': coverage,
            'CRPS': crps_approx
        })
    
    # Converter para DataFrame
    metrics_df = pd.DataFrame(results)
    
    # Adicionar média
    metrics_df.loc['Mean'] = metrics_df.mean()
    metrics_df.loc['Mean', 'Sample'] = 'Mean'
    
    return metrics_df

# Calcular métricas para as amostras visualizadas
try:
    from scipy.stats import norm
    metrics_df = calculate_uncertainty_metrics(samples)
    print(metrics_df)
except Exception as e:
    print(f"Erro ao calcular métricas: {e}")

## 7. Salvamento do Modelo e Exportação

Vamos salvar o modelo treinado para uso posterior.

### 7.1 Salvamento do Modelo

In [ ]:
# Criar diretório para o modelo
model_dir = os.path.join(OUTPUT_DIR, "model")
os.makedirs(model_dir, exist_ok=True)

# Criar diretório para checkpoints
ckpt_dir = os.path.join(OUTPUT_DIR, "checkpoints")
os.makedirs(ckpt_dir, exist_ok=True)

# Salvamento correto de checkpoint usando Lightning
ckpt_path = os.path.join(ckpt_dir, "crypto_moirai_moe_bayesian.ckpt")
trainer.save_checkpoint(ckpt_path)
print(f"Checkpoint Lightning salvo em {ckpt_path}")

# Salvamento adicional do estado do modelo (para compatibilidade)
model_path = os.path.join(model_dir, "crypto_moirai_moe_bayesian.pt")
torch.save(pl_module.state_dict(), model_path)
print(f"State dict salvo em {model_path}")

# Salvar apenas o modelo base (sem o Lightning Module)
base_model_path = os.path.join(model_dir, "crypto_moirai_moe_bayesian_base.pt")
torch.save(model.state_dict(), base_model_path)
print(f"Modelo base salvo em {base_model_path}")

print(f"\n✅ Modelos salvos usando padrão Lightning SOTA")

### 7.2 Salvamento da Configuração

In [ ]:
# Salvar a configuração utilizada
config_path = os.path.join(model_dir, "config.yaml")
with open(config_path, "w") as f:
    yaml.dump(config, f)

print(f"Configuração salva em {config_path}")

### 7.3 Exportação de Artefatos para Download

In [ ]:
# Comprimir artefatos importantes para download
import zipfile
import glob

# Criar arquivo ZIP com os artefatos principais
zip_path = os.path.join(BASE_DIR, "crypto_moirai_moe_bayesian_artifacts.zip")
with zipfile.ZipFile(zip_path, "w") as zipf:
    # Adicionar modelo e configuração
    zipf.write(model_path, os.path.basename(model_path))
    zipf.write(base_model_path, os.path.basename(base_model_path))
    zipf.write(config_path, os.path.basename(config_path))
    
    # Adicionar logs
    for log_file in glob.glob(os.path.join(LOG_DIR, "**/*.csv"), recursive=True):
        zipf.write(log_file, os.path.join("logs", os.path.basename(log_file)))
    
    # Adicionar gráficos
    for plot_file in glob.glob(os.path.join(PLOT_DIR, "**/*.png"), recursive=True):
        zipf.write(plot_file, os.path.join("plots", os.path.basename(plot_file)))

print(f"Artefatos comprimidos em {zip_path}")
print(f"Faça o download deste arquivo para uso posterior.")

## Resumo e Próximos Passos

Neste notebook, realizamos o fine-tuning do modelo Moirai-MoE com componentes bayesianos para previsão de séries temporais de criptomoedas usando o repositório uni2ts. Os principais passos foram:

1. Configuração do ambiente Kaggle
2. Download e instalação do repositório uni2ts
3. Preparação dos dados de criptomoedas da Binance
4. Configuração do modelo Moirai-MoE com componentes bayesianos
5. Treinamento e monitoramento do modelo usando o `MoiraiBayesianLightningModule` existente no projeto
6. Avaliação do modelo e visualização de resultados com intervalos de confiança
7. Salvamento do modelo e exportação de artefatos

### Benefícios de Usar a Implementação Existente

Neste notebook, utilizamos a classe `MoiraiBayesianLightningModule` do projeto uni2ts em vez de redefinir nossa própria implementação. Isso traz vários benefícios:

1. **Consistência com o projeto**: Garantimos que nosso treinamento segue exatamente a mesma lógica do projeto original.
2. **Redução de bugs**: Evitamos possíveis erros ao reimplementar uma lógica já testada e validada.
3. **Manutenção facilitada**: Se o projeto original for atualizado, podemos facilmente incorporar essas melhorias.
4. **Reprodutibilidade**: Os resultados são mais consistentes com os benchmarks oficiais.

### Próximos Passos

Para continuar o desenvolvimento do modelo, você pode considerar:

1. **Ajuste de Hiperparâmetros**: Experimente diferentes configurações para melhorar o desempenho do modelo.
2. **Adicionar Mais Ativos**: Expanda o conjunto de dados com mais criptomoedas para melhorar a generalização.
3. **Adicionar Features Externas**: Incorpore indicadores econômicos, sentimento de mercado, ou outros dados relevantes.
4. **Otimização do Modelo**: Experimente diferentes arquiteturas e componentes no modelo Moirai-MoE.
5. **Deploy do Modelo**: Implemente o modelo em produção para previsão contínua.

### Referências

- [Repositório uni2ts](https://github.com/waldefran/uni2ts)
- [Documentação do PyTorch Lightning](https://lightning.ai/docs/pytorch/stable/)
- [Tutorial de Fine-tuning no Kaggle](https://github.com/waldefran/uni2ts/blob/main/KAGGLE_FINETUNING_TUTORIAL.md)